1.	Structured Format Prompting: Instruct the model to output information as bullet lists and Markdown tables (e.g., “List three benefits of daily exercise in a Markdown table with columns ‘Benefit’ and ‘Description.’”); verify the output matches the requested structure.

In [1]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

def get_groq_response(prompt):
    """
    Sends a prompt to the Groq API using the Llama3-8b model.
    """
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0, # Low temperature for consistent formatting
    )
    return completion.choices[0].message.content.strip()

def verify_structure(content, format_type):
    """
    A basic verification step to check if the output matches the requested structure.
    """
    if format_type == "table":
        # Check for standard Markdown table indicators: pipes and hyphens
        return "|" in content and "-|-" in content or "---" in content
    elif format_type == "list":
        # Check for common bullet point markers
        bullet_markers = ["*", "-", "•", "1."]
        return any(content.strip().startswith(marker) for marker in bullet_markers) or "\n-" in content
    return False

def run_structured_prompting_lab():
    print("--- Lab Experiment: Structured Format Prompting ---")

    # Part A: Markdown Table Prompting
    # As specified in the source: "List three benefits of daily exercise in a Markdown table..."
    table_prompt = "List three benefits of daily exercise in a Markdown table with columns 'Benefit' and 'Description.'"
    print(f"\nStep 1: Requesting Markdown Table...")
    table_output = get_groq_response(table_prompt)
    print("Response Received:")
    print(table_output)
    
    is_table_valid = verify_structure(table_output, "table")
    print(f"Verification: {'PASSED' if is_table_valid else 'FAILED'} (Markdown table structure detected)")

    # Part B: Bulleted List Prompting
    list_prompt = "List three healthy snacks in a simple bulleted list format."
    print(f"\nStep 2: Requesting Bulleted List...")
    list_output = get_groq_response(list_prompt)
    print("Response Received:")
    print(list_output)
    
    is_list_valid = verify_structure(list_output, "list")
    print(f"Verification: {'PASSED' if is_list_valid else 'FAILED'} (Bullet list structure detected)")

if __name__ == "__main__":
    run_structured_prompting_lab()

--- Lab Experiment: Structured Format Prompting ---

Step 1: Requesting Markdown Table...
Response Received:
### Benefits of Daily Exercise
| Benefit | Description |
| --- | --- |
| Weight Management | Regular physical activity helps burn calories and maintain a healthy weight, reducing the risk of obesity and related diseases. |
| Improved Mental Health | Daily exercise releases endorphins, which can boost mood, reduce stress, and alleviate symptoms of anxiety and depression. |
| Increased Energy | Exercise can increase energy levels by improving sleep quality, enhancing cardiovascular health, and strengthening muscles, making it easier to tackle daily tasks. |
Verification: PASSED (Markdown table structure detected)

Step 2: Requesting Bulleted List...
Response Received:
* Apples
* Carrots
* Almonds
Verification: PASSED (Bullet list structure detected)


2.	JSON/YAML Generation: Provide a brief dataset description (e.g., three books with title, author, publication year) and prompt the model to produce valid JSON or YAML; use a parser to validate syntax and refine the prompt if errors occur.

In [5]:
import os
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
# Initialize the Groq client
# The SDK automatically draws from os.environ.get("GROQ_API_KEY")

def generate_and_validate_json(dataset_description: str, max_retries: int = 3) -> dict:
    """
    Sends a dataset description to Groq to generate a valid JSON object.
    Includes a parser-driven self-correction refinement loop if syntax errors occur.
    """
    
    # 1. Craft the initial baseline prompt
    user_prompt = (
        f"Generate a valid JSON array containing exactly the data described here: {dataset_description}. "
        "Each entry must follow a standardized structure based on the description provided."
    )
    
    # Maintain message thread context for iterative refinement
    messages = [
        {
            "role": "system",
            "content": "You are a helpful data transformation assistant. You must output raw JSON only. Do not include markdown codeblocks or wrapping formatting."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    
    attempt = 0
    while attempt < max_retries:
        attempt += 1
        print(f"\n[Attempt {attempt}] Requesting JSON from Groq...")
        
        try:
            # 2. Call the Groq Chat Completions API
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=messages,
                temperature=0.2, # Lower temperature for strictly deterministic formatting
                response_format={"type": "json_object"}  # Native Groq JSON Mode enforcement
            )
            
            raw_content = response.choices[0].message.content
            print(f"Received Response:\n{raw_content}")
            
            # 3. Parser Step: Validate JSON syntax locally
            parsed_json = json.loads(raw_content)
            print("Success! Syntax parser validated the JSON output cleanly.")
            return parsed_json
            
        except json.JSONDecodeError as err:
            print(f"Syntax Error Detected by Parser: {err}")
            
            if attempt >= max_retries:
                print("Maximum refinement attempts exhausted. Task failed.")
                raise err
                
            # 4. Refinement Loop: Append the parser error details and ask the model to self-correct
            print("Refining prompt with error details and requesting structural fix...")
            
            # Append the bad output so the model understands its previous state
            messages.append({"role": "assistant", "content": raw_content})
            
            # Append instructions containing the exact structural exception trace
            messages.append({
                "role": "user", 
                "content": f"The JSON string you provided is invalid and threw a JSONDecodeError: {str(err)}. Please fix the syntax constraints and output valid JSON."
            })

# --- Lab Demonstration Execution ---
if __name__ == "__main__":
    # Exercise objective description from the lab sheet
    lab_dataset_task = "three books with title, author, and publication year"
    
    print(f"--- Launching Lab LLM Task Demonstration ---")
    print(f"Target Objective: Transform raw description into validated JSON.")
    
    try:
        final_output = generate_and_validate_json(lab_dataset_task)
        print("\n--- Final Validated Output Result ---")
        print(json.dumps(final_output, indent=4))
        
    except Exception as e:
        print(f"\nLab execution crashed out: {e}")


--- Launching Lab LLM Task Demonstration ---
Target Objective: Transform raw description into validated JSON.

[Attempt 1] Requesting JSON from Groq...
Received Response:
{
  "books": [
       {
           "title": "To Kill a Mockingbird",
           "author": "Harper Lee",
           "publicationYear": 1960
       },
       {
           "title": "1984",
           "author": "George Orwell",
           "publicationYear": 1949
       },
       {
           "title": "Pride and Prejudice",
           "author": "Jane Austen",
           "publicationYear": 1813
       }
   ]
}
Success! Syntax parser validated the JSON output cleanly.

--- Final Validated Output Result ---
{
    "books": [
        {
            "title": "To Kill a Mockingbird",
            "author": "Harper Lee",
            "publicationYear": 1960
        },
        {
            "title": "1984",
            "author": "George Orwell",
            "publicationYear": 1949
        },
        {
            "title": "Pride and P

3.	Chain-of-Thought & Task Decomposition: Present a multi-step problem (e.g., a logic puzzle) and apply zero-shot CoT prompting (e.g., “Let’s think step by step. Explain your reasoning before the final answer.”); separately, decompose the problem into sequential sub-questions, collect partial answers, combine them, and compare accuracy against a direct-answer baseline.

In [8]:
import os
import json
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)
MODEL_NAME="llama-3.3-70b-versatile"
# A classic multi-step logic/math puzzle for the lab experiment
LOGIC_PUZZLE = (
    "A farmer has 15 sheep. A wolf attacks and kills all but 8 of them. "
    "The farmer then buys 5 more healthy sheep. How many live sheep does the farmer have now?"
)

def run_groq_inference(system_prompt: str, user_prompt: str) -> str:
    """Helper utility to communicate with Groq completions API."""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0  # Zero temperature for consistent lab reproduction
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"API Error: {str(e)}"

# =====================================================================
# PIPELINE 1: Direct-Answer Baseline
# =====================================================================
def run_baseline_pipeline():
    print("\n--- Running Method 1: Direct-Answer Baseline ---")
    system_prompt = "You are a precise calculator. Output ONLY the final numerical answer. No sentences."
    result = run_groq_inference(system_prompt, LOGIC_PUZZLE)
    print(f"Direct Baseline Output: {result}")
    return result

# =====================================================================
# PIPELINE 2: Zero-Shot Chain-of-Thought (CoT)
# =====================================================================
def run_zero_shot_cot_pipeline():
    print("\n--- Running Method 2: Zero-Shot Chain-of-Thought ---")
    system_prompt = "You are an analytical logic solver."
    # Adding the canonical zero-shot CoT trigger phrase
    cot_prompt = f"{LOGIC_PUZZLE}\n\nLet's think step by step. Explain your reasoning thoroughly before providing the final answer."
    result = run_groq_inference(system_prompt, cot_prompt)
    print(f"Zero-Shot CoT Output:\n{result}")
    return result

# =====================================================================
# PIPELINE 3: Programmatic Task Decomposition
# =====================================================================
def run_decomposition_pipeline():
    print("\n--- Running Method 3: Sequential Task Decomposition ---")
    system_prompt = "You are a precise fact-checker and calculator."
    
    # Define the broken-down sequential sub-questions
    sub_questions = [
        "Question 1: In the sentence 'A wolf attacks and kills all but 8 of them', how many sheep survived the wolf attack?",
        "Question 2: Based on the previous answer, if the farmer now has that many surviving sheep and buys 5 more, what is the new total of live sheep?"
    ]
    
    context = f"Context Context:\n{LOGIC_PUZZLE}\n\n"
    partial_answers = []
    
    # Sequentially execute sub-questions, passing previous answers forward
    for i, question in enumerate(sub_questions, 1):
        prompt = f"{context}\nFollowing up on our breakdown:\n{question}"
        print(f"-> Querying Sub-Question {i}...")
        
        answer = run_groq_inference(system_prompt, prompt)
        print(f"   Response {i}: {answer}\n")
        
        # Collect and append to history context loop
        partial_answers.append(f"Step {i} Result: {answer}")
        context += f"\n{partial_answers[-1]}"
        
    return partial_answers[-1]

# =====================================================================
# Execution & Analysis Engine
# =====================================================================
if __name__ == "__main__":
    print("=====================================================================")
    print("      LAB EXPERIMENT: CoT vs. DECOMPOSITION VS BASELINE PERFORMANCE  ")
    print("=====================================================================")
    print(f"Target Problem:\n\"{LOGIC_PUZZLE}\"")
    
    # Execute all three experimental setups
    baseline_res = run_baseline_pipeline()
    cot_res      = run_zero_shot_cot_pipeline()
    decomp_res   = run_decomposition_pipeline()
    
    print("=====================================================================")
    print("                     LAB SUMMARY & COMPARISON                        ")
    print("=====================================================================")
    print(f"1. Direct Baseline Output      : {baseline_res}")
    print(f"2. Zero-Shot CoT Final Verdict : (Check reasoning output box above)")
    print(f"3. Decomposition Final Answer  : {decomp_res.split('\n')[-1]}")
    print("=====================================================================")
    

      LAB EXPERIMENT: CoT vs. DECOMPOSITION VS BASELINE PERFORMANCE  
Target Problem:
"A farmer has 15 sheep. A wolf attacks and kills all but 8 of them. The farmer then buys 5 more healthy sheep. How many live sheep does the farmer have now?"

--- Running Method 1: Direct-Answer Baseline ---
Direct Baseline Output: 8 + 5 = 13

--- Running Method 2: Zero-Shot Chain-of-Thought ---
Zero-Shot CoT Output:
To solve this problem, let's break it down into steps and analyze the information given.

1. **Initial Number of Sheep**: The farmer starts with 15 sheep.

2. **Wolf Attack**: A wolf attacks and kills all but 8 of the sheep. This means that out of the 15 sheep, 8 survived the attack. To find out how many sheep were killed, we subtract the number of surviving sheep from the initial number:
   - Initial sheep = 15
   - Surviving sheep = 8
   - Sheep killed by the wolf = Initial sheep - Surviving sheep = 15 - 8 = 7

   So, 7 sheep were killed by the wolf, leaving 8 sheep alive.

3. **Buying 